In [ ]:
# Cell 0: Install dependencies (SageMaker / fresh kernel)
# Rerun this cell if you switch kernels.

import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

# Core deps (include LightGBM for Phase-2)
!{sys.executable} -m pip install -U pip
!{sys.executable} -m pip install -U polars pyarrow s3fs lightgbm shap scikit-learn pyyaml

# Install this repo in editable mode
!{sys.executable} -m pip install -e {ROOT}

print("Installed deps. ROOT=", ROOT)

# End-to-end Feature Analytics (Phase 0 → Phase 5)

This notebook runs the pipeline end-to-end and is resilient to "kernel status: unknown" by writing logs to disk.

## Phases covered
- **Phase 0**: Install deps + bootstrap contracts
- **Phase 1**: Build prediction dump + permutation importance + residual dataset/slices
- **Phase 2**: Residual model + interaction mining
- **Phase 3**: Cross-candidate generation
- **Phase 4**: Adapter compatibility / integration readiness checks
- **Phase 5**: Run metadata + data quality outputs + reproducible output folder (`outputs/<run_id>/`)

## Logging
Every command writes a timestamped log file under `logs/` (stdout + stderr).

In [ ]:
# Shared setup + file-based logging helper

from pathlib import Path
import json
import os
import subprocess
import sys
import time

import pandas as pd
import yaml

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
CONFIG_PATH = ROOT / "configs" / "analytics.yaml"
TRAIN_CFG_PATH = ROOT.parent / "Two_tower" / "configs" / "train.yaml"

LOG_DIR = ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("CONFIG_PATH:", CONFIG_PATH)
print("TRAIN_CFG_PATH:", TRAIN_CFG_PATH)
print("LOG_DIR:", LOG_DIR)


def run_and_log(cmd, *, name: str):
    """Run command, stream output to a log file, and raise on failure."""

    ts = time.strftime("%Y%m%d_%H%M%S")
    log_path = LOG_DIR / f"{ts}_{name}.log"
    print("\nRunning:", " ".join(map(str, cmd)))
    print("Log:", log_path)

    env = os.environ.copy()
    p = subprocess.Popen(
        cmd,
        cwd=str(ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        bufsize=1,
    )

    assert p.stdout is not None
    with open(log_path, "w", encoding="utf-8") as f:
        for line in p.stdout:
            f.write(line)
            # Keep notebook readable: print only key progress lines.
            if line.startswith("[") or "ERROR" in line or "Traceback" in line:
                print(line.rstrip())

    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed (exit={rc}). See log: {log_path}")

    return log_path


def latest_run_dir(base_outputs: Path) -> Path | None:
    """Pick most recent outputs/<run_id>/ by mtime of run_metadata.json."""

    metas = sorted(base_outputs.glob("*/run_metadata.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not metas:
        return None
    return metas[0].parent

In [ ]:
# Phase 0: Bootstrap contracts (manifest + feature catalog)

inputs_dir = ROOT / "inputs"
inputs_dir.mkdir(parents=True, exist_ok=True)

with open(TRAIN_CFG_PATH, "r", encoding="utf-8") as f:
    train_cfg = yaml.safe_load(f)

features = train_cfg["features"]
user_features = list(features.get("user_feature_cols") or [])
client_features = list(features.get("client_feature_cols") or [])
user_multi = set(features.get("user_multi_cols") or [])
client_multi = set(features.get("client_multi_cols") or [])

rows = []
for c in user_features:
    rows.append({"feature_name": c, "tower": "user", "feature_type": "multi" if c in user_multi else "other", "is_multi": c in user_multi})
for c in client_features:
    rows.append({"feature_name": c, "tower": "client", "feature_type": "multi" if c in client_multi else "other", "is_multi": c in client_multi})

feature_catalog = pd.DataFrame(rows).drop_duplicates(subset=["feature_name"]).sort_values("feature_name")
feature_catalog_path = inputs_dir / "feature_catalog.parquet"
feature_catalog.to_parquet(feature_catalog_path, index=False)

manifest = {
    "model_id": "two_tower_mlp",
    "model_version": "v1",
    "trained_at": "2026-05-01T00:00:00Z",
    "label_column": features.get("label_col", "label"),
    "prediction_schema_version": "1.0.0",
    "feature_catalog_version": "1.0.0",
    "artifact_uris": {"artifacts_base": train_cfg["paths"]["artifacts_base"]},
}
manifest_path = inputs_dir / "model_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Wrote:", feature_catalog_path)
print("Wrote:", manifest_path)
print("Feature rows:", len(feature_catalog) )

In [ ]:
# Phase 1: Build prediction dump (from Two Tower validation data)

build_cmd = [sys.executable, "-m", "pipelines.build_prediction_dump", "--config", str(CONFIG_PATH)]
run_and_log(build_cmd, name="phase1_build_prediction_dump")

In [ ]:
# Phase 0.5: Validate contracts

validate_cmd = [sys.executable, "-m", "pipelines.validate_contracts", "--config", str(CONFIG_PATH)]
run_and_log(validate_cmd, name="phase0_validate_contracts")

In [ ]:
# Phase 1→5: Run the analytics pipeline (Phase‑1/2/3 + Phase‑5 outputs)
# This will write results under outputs/<run_id>/ ... and logs under logs/ ...

run_cmd = [sys.executable, "-m", "pipelines.run_mvp", "--config", str(CONFIG_PATH)]
run_and_log(run_cmd, name="phase1to5_run_mvp")

BASE_OUT = ROOT / "outputs"
RUN_DIR = latest_run_dir(BASE_OUT)
print("BASE_OUT:", BASE_OUT)
print("RUN_DIR:", RUN_DIR)
if RUN_DIR is None:
    raise RuntimeError("No outputs/<run_id>/run_metadata.json found. Pipeline may not have finished.")

In [ ]:
# Phase 1 review: Permutation importance + residual slices

pi_path = RUN_DIR / "feature_importance" / "permutation_importance.csv"
guard_path = RUN_DIR / "feature_importance" / "true_rescore_guard_report.json"
residual_slices_path = RUN_DIR / "residual_analysis" / "residual_slices.csv"

pi_df = pd.read_csv(pi_path)
residual_slices_df = pd.read_csv(residual_slices_path)
guard = json.loads(guard_path.read_text(encoding="utf-8"))

print("Run dir:", RUN_DIR)

print("\nTop 20 features by AUC drop")
display(pi_df.sort_values("auc_drop", ascending=False).head(20))

print("\nBottom 20 features by AUC drop (pruning candidates)")
display(pi_df.sort_values("auc_drop", ascending=True).head(20))

print("\nTop 20 weak slices by avg residual (filtered)")
display(residual_slices_df.sort_values("avg_residual_abs", ascending=False).head(20))

print("\nGuard report summary")
display(pd.DataFrame([
    {
        "mode": guard.get("mode"),
        "used_features": guard.get("used_feature_cols_count"),
        "ignored_requested": guard.get("ignored_requested_feature_cols_count"),
        "missing_scorable_features": guard.get("missing_scorable_feature_cols_count"),
        "missing_client_embedding_count": guard.get("missing_client_embedding_count"),
        "missing_client_embedding_ratio": guard.get("missing_client_embedding_ratio"),
    }
]))

In [ ]:
# Phase 2 review: Residual model metrics + feature importance + interactions

residual_metrics_path = RUN_DIR / "residual_analysis" / "residual_model_metrics.json"
tree_fi_path = RUN_DIR / "residual_analysis" / "tree_feature_importance.csv"
interaction_scores_path = RUN_DIR / "interactions" / "interaction_scores.csv"

if residual_metrics_path.exists():
    print("Residual model metrics")
    metrics = json.loads(residual_metrics_path.read_text(encoding="utf-8"))
    display(pd.DataFrame([metrics]))
else:
    print("Residual model metrics not found")

if tree_fi_path.exists():
    print("\nTop 30 residual-model features (gain)")
    tree_fi = pd.read_csv(tree_fi_path)
    display(tree_fi.sort_values("importance_gain", ascending=False).head(30))
else:
    print("Tree feature importance not found")

if interaction_scores_path.exists():
    print("\nTop 30 interaction candidates")
    inter_df = pd.read_csv(interaction_scores_path)
    display(inter_df.head(30))
else:
    print("Interaction scores not found (interaction mining may have failed or been disabled)")

In [ ]:
# Phase 3 review: Cross-candidate outputs

cross_yaml_path = RUN_DIR / "cross_candidates" / "candidate_crosses.yaml"
cross_meta_path = RUN_DIR / "cross_candidates" / "candidate_cross_metadata.csv"

if cross_yaml_path.exists():
    print("Candidate crosses (YAML)")
    print(cross_yaml_path.read_text(encoding="utf-8")[:2000])
else:
    print("Candidate crosses YAML not found")

if cross_meta_path.exists():
    print("\nCandidate cross metadata (top 30)")
    cross_meta = pd.read_csv(cross_meta_path)
    display(cross_meta.head(30))
else:
    print("Candidate cross metadata not found")

In [ ]:
# Phase 4 review: Adapter compatibility / integration readiness

compat = ROOT / "adapters" / "two_tower_adapter" / "COMPATIBILITY.md"
print("Two Tower adapter compatibility doc:", compat)
if compat.exists():
    print(compat.read_text(encoding="utf-8")[:2000])
else:
    print("COMPATIBILITY.md not found")

print("\nIgnored requested features (first 50):")
print((guard.get("ignored_requested_feature_cols") or [])[:50])

In [ ]:
# Phase 5 review: Run metadata + data quality

meta_path = RUN_DIR / "run_metadata.json"
dq_summary_path = RUN_DIR / "data_quality" / "summary.json"
dq_per_feature_path = RUN_DIR / "data_quality" / "per_feature.csv"

print("Run metadata:", meta_path)
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    display(pd.DataFrame([
        {
            "run_id": meta.get("run_id"),
            "created_at_utc": meta.get("created_at_utc"),
            "elapsed_s": (meta.get("outputs") or {}).get("elapsed_s"),
        }
    ]))
else:
    print("run_metadata.json not found")

print("\nData quality summary:", dq_summary_path)
if dq_summary_path.exists():
    dq = json.loads(dq_summary_path.read_text(encoding="utf-8"))
    display(pd.DataFrame([dq]))
else:
    print("data quality summary not found")

print("\nWorst 30 features by null rate:")
if dq_per_feature_path.exists():
    dq_pf = pd.read_csv(dq_per_feature_path)
    display(dq_pf.sort_values("null_rate", ascending=False).head(30))
else:
    print("per_feature.csv not found")